# 01 · Function calling을 수동으로 구현하기 · 45분

## 학습 목표
- 모델의 도구 **요청**과 Python의 도구 **실행**을 구분합니다.
- JSON Schema, `call_id`, 도구 결과 재전달을 직접 확인합니다.

## 개념
`질문 → 모델의 function_call → 허용된 Python 함수 → 외부 API → function_call_output → 최종 답변`
모델이 네트워크를 직접 호출하는 것이 아닙니다. 실행 권한은 애플리케이션이 갖습니다. 함수 이름을 `eval`로 실행하거나 모델이 만든 URL로 요청하지 않습니다.
이 노트북의 핵심은 모든 줄을 외우는 것이 아니라 **요청/실행/결과/최종 답변** 네 경계를 로그에서 찾는 것입니다.


In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


### 1. LLM 없이 도구를 먼저 시험하기
날씨는 Open-Meteo의 현재 값입니다. 환율은 Frankfurter의 최근 영업일 참고값이며 회사 정산 환율이나 은행 견적이 아닙니다. 장애가 나면 가짜 값으로 바꾸지 않고 오류를 표시합니다.

In [ ]:
from hanbit_assistant.tools import get_weather, convert_currency
print(json.dumps(await get_weather("서울"), ensure_ascii=False, indent=2))
print(json.dumps(await convert_currency(100, "USD", "KRW"), ensure_ascii=False, indent=2))


### 2. 모델에게 보여줄 도구 설명서
`description`은 언제 도구가 필요한지, `parameters`는 입력 구조를 설명합니다. `additionalProperties=False`와 필수 인수를 명시합니다.

In [ ]:
tool_schemas = [
    {"type": "function", "name": "get_weather", "description": "서울, 도쿄, 런던, 뉴욕의 현재 날씨 조회",
     "strict": True, "parameters": {"type": "object", "properties": {
         "city": {"type": "string", "description": "지원 도시 이름"}},
         "required": ["city"], "additionalProperties": False}},
    {"type": "function", "name": "convert_currency", "description": "최근 영업일 참고 환율로 금액 환산",
     "strict": True, "parameters": {"type": "object", "properties": {
         "amount": {"type": "number"}, "from_currency": {"type": "string"},
         "to_currency": {"type": "string"}},
         "required": ["amount", "from_currency", "to_currency"], "additionalProperties": False}},
]
allowed_functions = {"get_weather": get_weather, "convert_currency": convert_currency}
print([t["name"] for t in tool_schemas])


## 실험 · 예측 → 실행 → 관찰 → 변경
1. **예측:** 도쿄 날씨만 묻는다면 `get_weather`와 `convert_currency` 중 무엇이 필요할까요? `prediction`에 기록합니다.
2. **실행:** 아래 기본 질문/`TOOL_CHOICE="auto"`로 실행합니다. Python이 함수를 호출하기 **전** 요청 로그와 **후** 결과 로그를 구분하세요.
3. **관찰:** 실행 횟수와 실제 도구 결과의 기준 시각/기준일을 확인합니다. `call_id`는 요청과 결과를 연결하는 표식입니다.
4. **변경:** 질문을 그대로 두고 `TOOL_CHOICE="none"`만 바꿔 같은 셀을 다시 실행합니다. 실제 함수 실행 횟수가 0인지 확인하세요. 날씨를 말하더라도 검증된 최신 값이 아닙니다.

추가 연습은 질문만 “파리 현재 날씨”로 바꾸고 `auto`로 복원하기입니다. 지원하지 않는 도시는 확인 질문 또는 오류가 필요합니다.

### 수동 실행 루프 읽는 법
아래는 실제 모델과 외부 API를 호출합니다. 위의 함수 단독 실습과 달리 여기서는 모델이 도구를 **선택**합니다.
reasoning의 `encrypted_content`는 다음 호출을 이어가는 불투명한 데이터입니다. 숨겨진 추론으로 해석하거나 `history` 전체를 출력하지 않습니다.


In [ ]:
prediction = ""  # 실행할 함수 이름을 먼저 예상
QUESTION = "도쿄 현재 날씨를 알려줘."
TOOL_CHOICE = "auto"  # 변경 실험: "none". 두 실행에서 질문은 고정합니다.
assert TOOL_CHOICE in {"auto", "none"}
history = [{"role": "user", "content": QUESTION}]
executed = []
async with create_openai_client(settings) as client:
    for round_number in range(5):
        result = await client.responses.create(
            model=settings.model, input=history, tools=tool_schemas,
            parallel_tool_calls=False, tool_choice=TOOL_CHOICE, store=False, max_output_tokens=650,
            include=["reasoning.encrypted_content"],
        )
        calls = [item for item in result.output if item.type == "function_call"]
        if not calls:
            print("최종 답변:", result.output_text)
            break
        for item in result.output:
            if item.type in {"reasoning", "function_call"}:
                fields = ({"type", "id", "summary", "encrypted_content", "status"}
                          if item.type == "reasoning"
                          else {"type", "id", "call_id", "name", "arguments", "status"})
                history.append({k: v for k, v in item.model_dump(exclude_none=True).items() if k in fields})
        for call in calls:
            if call.name not in allowed_functions:
                raise ValueError("허용되지 않은 도구 요청")
            arguments = json.loads(call.arguments)
            print("도구 요청 (아직 실행 전):", call.name, arguments, "call_id:", call.call_id)
            tool_result = await allowed_functions[call.name](**arguments)
            executed.append(call.name)
            print("도구 결과:", json.dumps(tool_result, ensure_ascii=False))
            history.append({"type": "function_call_output", "call_id": call.call_id,
                            "output": json.dumps(tool_result, ensure_ascii=False)})
    else:
        raise RuntimeError("도구 반복 제한 초과: 무한 실행하지 않습니다.")

print("실제 함수 실행 횟수:", len(executed), "실행 목록:", executed)
if TOOL_CHOICE == "none":
    assert not executed, "도구 금지 조건에서는 실행하면 안 됩니다."


## 관찰 기록과 체크포인트
| 조건 | 예상 함수 | 실제 실행 목록/횟수 | 현재 값의 출처가 있는가? |
|---|---|---|---|
| auto | 직접 기록 | 직접 기록 | 직접 기록 |
| none | 직접 기록 | 직접 기록 | 직접 기록 |

- [ ] `function_call`은 요청이며 Python의 `await allowed_functions[...]`가 실행 경계임을 찾았다.
- [ ] `function_call_output`과 `call_id`로 결과가 모델에 돌아가는 위치를 찾았다.
- [ ] 최대 5회 반복 후 중단하는 이유를 설명했다.

**결과 질문:** 모델이 “조회했습니다”라고 말했지만 실제 함수 실행이 0회라면 믿을 수 있나요?
**연습:** 두 번째로 환율만 묻는 질문을 직접 만들고 도구 선택을 예측하세요. 한 번에 질문과 지침을 모두 바꾸지 않습니다.

<details><summary>참고 풀이 · 예시 해설 (예측/실행 뒤 열기)</summary>
`QUESTION="100 USD를 KRW로 환산하고 참고 환율 기준일을 알려줘."`, `TOOL_CHOICE="auto"`로 실행합니다.
일반적으로 `convert_currency`가 필요합니다. 모델 선택은 달라질 수 있으므로 로그로 확인합니다. `none`에서는 함수를 실행하지 않습니다. 함수 설명만 등록했다고 실제 데이터가 조회되지는 않습니다.
</details>

### 실패하면
`ToolError`는 지원 도시/통화/네트워크 문제인지 확인하세요. 결과를 가짜 날씨/환율로 대체하지 않습니다.
반복 제한이면 질문을 한 도구만 필요한 질문으로 줄입니다. `call_id` 관련 400이면 이전 history를 수동 편집하지 말고 이 셀 처음부터 다시 실행합니다.
01의 수동 반복을 다음 노트북에서 Framework로 교체합니다.
